In [63]:
import pandas as pd

df = pd.read_csv("SCMS_Delivery_History_Dataset.csv")
df.head()
#displays 5 columns of CSV

,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,...,Unit of Measure (Per Pack),Line Item Quantity,Line Item Value,Pack Price,Unit Price,Manufacturing Site,First Line Designation,Weight (Kilograms),Freight Cost (USD),Line Item Insurance (USD)
0,1,100-CI-T01,Pre-PQ Process,SCMS-4,ASN-8,Côte d'Ivoire,PMO - US,Direct Drop,EXW,Air,...,30,19,551.0,29.00,0.97,Ranbaxy Fine Chemicals LTD,Yes,13,780.34,NaN
1,3,108-VN-T01,Pre-PQ Process,SCMS-13,ASN-85,Vietnam,PMO - US,Direct Drop,EXW,Air,...,240,1000,6200.0,6.20,0.03,"Aurobindo Unit III, India",Yes,358,4521.5,NaN
2,4,100-CI-T01,Pre-PQ Process,SCMS-20,ASN-14,Côte d'Ivoire,PMO - US,Direct Drop,FCA,Air,...,100,500,40000.0,80.00,0.80,ABBVIE GmbH & Co.KG Wiesbaden,Yes,171,1653.78,NaN
3,15,108-VN-T01,Pre-PQ Process,SCMS-78,ASN-50,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,31920,127360.8,3.99,0.07,"Ranbaxy, Paonta Shahib, India",Yes,1855,16007.06,NaN
4,16,108-VN-T01,Pre-PQ Process,SCMS-81,ASN-55,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,38000,121600.0,3.20,0.05,"Aurobindo Unit III, India",Yes,7590,45450.08,NaN


In [64]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
#gives the dimensions of the dataset and the column titles

Shape: (10324, 33)

Columns:
['ID', 'Project Code', 'PQ #', 'PO / SO #', 'ASN/DN #', 'Country', 'Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'PQ First Sent to Client Date', 'PO Sent to Vendor Date', 'Scheduled Delivery Date', 'Delivered to Client Date', 'Delivery Recorded Date', 'Product Group', 'Sub Classification', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Dosage Form', 'Unit of Measure (Per Pack)', 'Line Item Quantity', 'Line Item Value', 'Pack Price', 'Unit Price', 'Manufacturing Site', 'First Line Designation', 'Weight (Kilograms)', 'Freight Cost (USD)', 'Line Item Insurance (USD)']


In [65]:
print(df['Freight Cost (USD)'].dtype)
print(df['Freight Cost (USD)'].unique()[:20]) 
#shows what variable freigh cost is stored as
#gives data for what might see under freight cost, need to convert to numeric

str
<StringArray>
[                            '780.34',                             '4521.5',
                            '1653.78',                           '16007.06',
                           '45450.08',                            '5920.42',
 'Freight Included in Commodity Cost',                            '6212.41',
              'See ASN-93 (ID#:1281)',                           '13569.49',
                'Invoiced Separately',                           '64179.42',
                            '1760.32',                             '3120.7',
                             '912.96',                            '2682.47',
                           '15893.71',                            '4193.49',
                            '1767.38',                            '3518.38']
Length: 20, dtype: str


In [66]:
print(df['Shipment Mode'].value_counts())
print(df['Line Item Value'].dtype) 
#gives the number of each mode of shipment

Shipment Mode
Air            6113
Truck          2830
Air Charter     650
Ocean           371
Name: count, dtype: int64
float64


In [67]:
#Freight Cost (USD) is stored as text, not numbers. Based on the unique values, most entries are real dollar amounts, 
#but some are notes like "Freight Included in Commodity Cost" or "Invoiced Separately" instead of a number. 
#I'm using errors='coerce' so pandas converts what it can and turns the rest into NaN, rather than crashing or forcing me to guess at a numeric value

In [68]:
df['Freight Cost Clean'] = pd.to_numeric(df['Freight Cost (USD)'], errors='coerce')

print("Successfully converted:", df['Freight Cost Clean'].notna().sum())
print("Could not convert (became NaN):", df['Freight Cost Clean'].isna().sum())
#converts all number strings to numeric, and adds to the count of total shipments with a numeric entry under freigh cost

Successfully converted: 6198
Could not convert (became NaN): 4126


In [69]:
#Could not convert about 40% of the data, which is way more than I expected for a "few placeholder values" — worth understanding why before I just drop it. 
#Based on the text values, the unable to convert freight costs are either freight bundled into the commodity price, freight invoiced on a separate document, 
#or freight recorded against a different line item in a consolidated shipment. 

In [70]:
print(df.loc[df['Freight Cost Clean'].isna(), 'Freight Cost (USD)'].value_counts()) 
#gives number of each non numeric entry

Freight Cost (USD)
Freight Included in Commodity Cost    1442
Invoiced Separately                    239
See DN-304 (ID#:10589)                  16
See ASN-32231 (ID#:13648)               14
See ASN-31750 (ID#:19272)               14
                                      ... 
See DN-4153 (ID#:86170)                  1
See DN-4259 (ID#:86808)                  1
See DN-4265 (ID#:83335)                  1
See DN-4274 (ID#:84472)                  1
See DN-4282 (ID#:83919)                  1
Name: count, Length: 1301, dtype: int64


In [71]:
# Although none of these freight costs that could not convert can be categorized as missing data, 
# I am going to create a new dataset without the not directly reported freight costs because I do not have access to the other data set that are 
# referenced. This means my analysis only reflects the ~60% of shipments where freight 
# cost was directly and separately reported - the findings may not fully 
# generalize to shipments where freight is bundled into pricing differently.

In [72]:
df_clean = df[df['Freight Cost Clean'].notna()].copy()
print(f"Working dataset: {len(df_clean)} rows out of {len(df)} original rows ({len(df_clean)/len(df)*100:.1f}%)")
#removed all rows that do not have a numeric entry under freigh cost clean

Working dataset: 6198 rows out of 10324 original rows (60.0%)


In [73]:
df_clean['freight_pct_of_value'] = (df_clean['Freight Cost Clean'] / df_clean['Line Item Value']) * 100

print(df_clean['freight_pct_of_value'].describe()) 
#gives shipment cost as a percentage of freight value

count    6198.000000
mean             inf
std              NaN
min         0.011189
25%         4.418850
50%        10.592548
75%        28.735864
max              inf
Name: freight_pct_of_value, dtype: float64


C:\Users\cetha\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [74]:
print("Rows with zero Line Item Value:", (df_clean['Line Item Value'] == 0).sum())
print("Rows with negative Line Item Value:", (df_clean['Line Item Value'] < 0).sum())

df_clean[df_clean['Line Item Value'] <= 0][['ID', 'Vendor', 'Country', 'Line Item Value', 'Freight Cost Clean']] 
#gave the total number of shipments with line item 0, to know how prominent this is

Rows with zero Line Item Value: 12
Rows with negative Line Item Value: 0


,ID,Vendor,Country,Line Item Value,Freight Cost Clean
1284,10910,SCMS from RDC,Côte d'Ivoire,0.0,2951.94
2398,12522,SCMS from RDC,Mozambique,0.0,2765.64
2750,14079,"Standard Diagnostics, Inc.",Tanzania,0.0,20583.37
2915,17135,"Standard Diagnostics, Inc.",Tanzania,0.0,21384.75
3023,19405,"Standard Diagnostics, Inc.",Tanzania,0.0,21351.18
3338,24653,"Standard Diagnostics, Inc.",Tanzania,0.0,21384.75
5128,56899,"Standard Diagnostics, Inc.",Tanzania,0.0,16382.89
5362,61493,"Standard Diagnostics, Inc.",Dominican Republic,0.0,1428.23
5628,65555,"Standard Diagnostics, Inc.",Tanzania,0.0,20906.77
6057,72922,"Standard Diagnostics, Inc.",Tanzania,0.0,25778.90


In [75]:
#Once I calculated freight cost as a percentage of Line Item Value, I got inf (infinity) for the mean and max, 
#which meant something was being divided by zero. A few rows have Line Item Value listed as exactly 0, 
#so dividing freight cost by that produces infinity. I'm removing these rows rather than trying to fill in a value, 
#since I have no reliable way to guess what the actual line item value should have been.

In [76]:
df_clean = df_clean[df_clean['Line Item Value'] > 0].copy()

df_clean['freight_pct_of_value'] = (df_clean['Freight Cost Clean'] / df_clean['Line Item Value']) * 100

print(df_clean['freight_pct_of_value'].describe()) #removes shipments with line item values of 0
#line item values of 0 are either a data entry error, or a donation of some kind

count    6.186000e+03
mean     2.547924e+03
std      1.711746e+05
min      1.118869e-02
25%      4.409253e+00
50%      1.056458e+01
75%      2.857211e+01
max      1.344940e+07
Name: freight_pct_of_value, dtype: float64


In [77]:
df_clean.sort_values('freight_pct_of_value', ascending=False)[
    ['ID', 'Vendor', 'Country', 'Line Item Value', 'Freight Cost Clean', 'freight_pct_of_value']
].head(15)  
#this points out the 15 largest outliers in shipment cost as a percentage of freight value

,ID,Vendor,Country,Line Item Value,Freight Cost Clean,freight_pct_of_value
4599,47301,MYLAN LABORATORIES LTD (FORMERLY MATRIX LABORA...,Zimbabwe,0.01,1344.94,1.344940e+07
8853,85009,SCMS from RDC,Nigeria,18.44,69737.00,3.781833e+05
1113,10679,SCMS from RDC,Kenya,0.03,99.00,3.300000e+05
3356,24990,"Trinity Biotech, Plc",Namibia,0.25,751.43,3.005720e+05
5393,61882,Aurobindo Pharma Limited,South Africa,5.06,4116.81,8.135988e+04
9881,86277,SCMS from RDC,Nigeria,156.16,124350.82,7.963039e+04
1096,10660,SCMS from RDC,Rwanda,20.93,12977.28,6.200325e+04
8348,84409,SCMS from RDC,Nigeria,48.66,26731.00,5.493424e+04
1511,11236,SCMS from RDC,Guyana,1.52,798.59,5.253882e+04
1716,11547,SCMS from RDC,Côte d'Ivoire,2.50,1225.42,4.901680e+04


In [78]:
# Even after removing the zero-value rows, the max freight percentage was still over 13 million percent, which isn't realistic. 
# Looking at the worst offenders, they mostly have a Line Item Value of a few cents while freight cost is a normal dollar amount. 
# I'm using a 99th percentile cutoff instead of picking an arbitrary number by eye, 
# since it's a defensible, repeatable way to remove extreme values without me subjectively deciding where "too high" starts.

In [79]:
cutoff = df_clean['freight_pct_of_value'].quantile(0.99)
print(f"99th percentile cutoff: {cutoff:.1f}%")

df_clean = df_clean[df_clean['freight_pct_of_value'] <= cutoff].copy()
print(df_clean['freight_pct_of_value'].describe()) 
#cleans those top 1% of outliers from the dataset

99th percentile cutoff: 3183.8%
count    6124.000000
mean       50.274852
std       169.777898
min         0.011189
25%         4.373380
50%        10.410568
75%        27.553034
max      3152.163446
Name: freight_pct_of_value, dtype: float64


In [80]:
# Now that the data's cleaned up, the first question I want to answer is whether shipment mode (Air, Truck, Ocean, Air Charter) 
# actually explains differences in freight cost. Grouping by mode and comparing the median freight percentage is the most direct way to check that.

In [81]:
mode_summary = df_clean.groupby('Shipment Mode')['freight_pct_of_value'].agg(['mean', 'median', 'count'])
print(mode_summary.sort_values('mean', ascending=False)) 
#gives the mean, median, and count of each type of shipping method

                    mean     median  count
Shipment Mode                             
Air Charter    63.765190   8.435131    417
Air            56.561772  12.433068   4057
Truck          32.756726   5.824106   1161
Ocean          30.603468   3.469580    281


In [82]:
mode_summary_full = df_clean.groupby('Shipment Mode')['freight_pct_of_value'].describe()
print(mode_summary_full[['count', 'mean', '50%', '75%', 'max']]) 
#gives the count, mean, median, Q3, and max of each shipping method

                count       mean        50%        75%          max
Shipment Mode                                                      
Air            4057.0  56.561772  12.433068  34.034536  2945.750000
Air Charter     417.0  63.765190   8.435131  26.152736  2244.178295
Ocean           281.0  30.603468   3.469580   8.865207  3152.163446
Truck          1161.0  32.756726   5.824106  15.838333  3089.224852


In [83]:
# The mean freight percentage is much higher than the median in every single mode, which tells me each mode has its own outlier shipments 
# pulling the average up. Looking at the 75th percentile instead of just the mean confirms Air and Air Charter 
# are still consistently more expensive at every level, not just skewed by a few outliers, which is a more reliable way to state the finding.

In [84]:
# Mode clearly matters, but there's still a lot of spread within each mode, which means another factor is liekly also driving cost. 
# Vendor is the next obvious variable to check, since different suppliers might negotiate different freight rates or have different logistics setups.

In [85]:
top_vendors = df_clean['Vendor'].value_counts().head(10).index
vendor_summary_ex = df_clean[
    (df_clean['Vendor'].isin(top_vendors)) & (df_clean['Vendor'] != 'SCMS from RDC')
].groupby('Vendor')['freight_pct_of_value'].agg(['mean', 'median', 'count'])
print(vendor_summary_ex.sort_values('median', ascending=False)) 
#remove SCMS from RDC as this is like multiple vendors under 1 name

                                                         mean     median  \
Vendor                                                                     
ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV)     47.168367  25.049440   
Trinity Biotech, Plc                                49.739560  15.264500   
Standard Diagnostics, Inc.                          83.037371  14.628160   
Aurobindo Pharma Limited                            55.883199   9.905215   
HETERO LABS LIMITED                                 15.426833   9.769830   
CHEMBIO DIAGNOSTIC SYSTEMS, INC.                    19.613624   9.667698   
Orgenics, Ltd                                       29.260978   9.154637   
CIPLA LIMITED                                       20.119605   7.258162   
MYLAN LABORATORIES LTD (FORMERLY MATRIX LABORAT...  26.602738   5.774055   

                                                    count  
Vendor                                                     
ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS

In [86]:
# "SCMS from RDC" shows up as a "vendor" with over 3,000 shipments,  
# more than half the entire cleaned dataset, while every other vendor has a few hundred at most. 
# Based on the name (Regional Distribution Center), this is likely an internal distribution hub rather than an actual external supplier, 
# so comparing it directly against real vendors isn't a fair comparison. I'm excluding it from the vendor analysis so the comparison 
# is between actual supplier relationships.

In [87]:
top_vendors = df_clean['Vendor'].value_counts().head(10).index

vendor_summary = df_clean[df_clean['Vendor'].isin(top_vendors)].groupby('Vendor')['freight_pct_of_value'].agg(['mean', 'median', 'count'])
print(vendor_summary.sort_values('mean', ascending=False))
#gives the mean and median of shipping cost as a percentage of freight value
#with the 10 highest volume vendors to help eliminate outlieers

                                                         mean     median  \
Vendor                                                                     
Standard Diagnostics, Inc.                          83.037371  14.628160   
SCMS from RDC                                       60.053807   9.921026   
Aurobindo Pharma Limited                            55.883199   9.905215   
Trinity Biotech, Plc                                49.739560  15.264500   
ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV)     47.168367  25.049440   
Orgenics, Ltd                                       29.260978   9.154637   
MYLAN LABORATORIES LTD (FORMERLY MATRIX LABORAT...  26.602738   5.774055   
CIPLA LIMITED                                       20.119605   7.258162   
CHEMBIO DIAGNOSTIC SYSTEMS, INC.                    19.613624   9.667698   
HETERO LABS LIMITED                                 15.426833   9.769830   

                                                    count  
Vendor                     

In [88]:
df_clean.to_csv("freight_analysis_clean.csv", index=False)
print("Saved:", df_clean.shape) 
#new clean data set used to create charts and make decisions 

Saved: (6124, 35)


In [89]:
df_clean['lane'] = df_clean['Manufacturing Site'] + " → " + df_clean['Country']

print("Unique manufacturing sites:", df_clean['Manufacturing Site'].nunique())
print("Unique destination countries:", df_clean['Country'].nunique())
print("Unique lanes:", df_clean['lane'].nunique())
# Lane analysis: checking if specific origin-destination combinations 
# ("lanes") show meaningfully different freight costs, beyond what 
# mode or vendor alone explain.

Unique manufacturing sites: 76
Unique destination countries: 39
Unique lanes: 466


In [90]:
lane_counts = df_clean['lane'].value_counts()
print(lane_counts.describe())
print("\nLanes with 30+ shipments:", (lane_counts >= 30).sum())
print("Lanes with 20+ shipments:", (lane_counts >= 20).sum())
print("Lanes with 10+ shipments:", (lane_counts >= 10).sum())
# Checking how many shipments each lane actually has, since with 466 
# unique lanes and ~6,100 shipments, most lanes will have very few 
# shipments each - not enough to trust a median calculated from them.

count    466.000000
mean      13.141631
std       26.706717
min        1.000000
25%        1.000000
50%        4.000000
75%       13.000000
max      237.000000
Name: count, dtype: float64

Lanes with 30+ shipments: 54
Lanes with 20+ shipments: 83
Lanes with 10+ shipments: 140


In [91]:
high_volume_lanes = lane_counts[lane_counts >= 30].index

lane_summary = df_clean[df_clean['lane'].isin(high_volume_lanes)].groupby('lane')['freight_pct_of_value'].agg(['mean', 'median', 'count'])
print(lane_summary.sort_values('median', ascending=False).head(15))
print("\n...")
print(lane_summary.sort_values('median', ascending=False).tail(15))
# Using a 30+ shipment threshold gives 54 lanes with a real enough 
# sample size to compare, without diluting into statistical noise 
# from lanes with only a handful of shipments.

                                                        mean     median  count
lane                                                                          
Aurobindo Unit III, India → Guyana                319.958397  88.218220     48
Aurobindo Unit III, India → Haiti                 181.886349  85.883297    127
ABBVIE Ludwigshafen Germany → Haiti                99.665852  37.301327     30
KHB Test Kit Facility, Shanghai China → Ethiopia   29.196665  27.716540     32
Aurobindo Unit III, India → Vietnam               112.889916  24.144230     86
Aurobindo Unit III, India → Côte d'Ivoire          80.544768  23.999315    237
ABBVIE Ludwigshafen Germany → Uganda               25.207582  21.414201     37
Aurobindo Unit III, India → Zimbabwe               51.818557  17.949502     48
Cipla, Goa, India → Uganda                         91.669622  17.213149     31
Aurobindo Unit III, India → Rwanda                 56.777670  16.790327     83
Strides, Bangalore, India. → Vietnam               3

In [92]:
# I noticed that several of the vendors with higher freight costs were also shipping mostly by air, 
# which raised the question of whether destination could be doing something similar — 
#maybe certain countries are just harder or more expensive to ship to regardless of who's shipping or how. 
# Checking freight cost by destination country is the next variable to test, especially since Haiti stood out early on.

In [93]:
countries_of_interest = ['Haiti', 'Rwanda', 'Zambia', 'Zimbabwe', 'South Africa', 'Vietnam']
country_summary = df_clean[df_clean['Country'].isin(countries_of_interest)].groupby('Country')['freight_pct_of_value'].agg(['mean', 'median', 'count'])
print(country_summary.sort_values('median', ascending=False))

                    mean     median  count
Country                                   
Haiti         106.129143  28.894716    389
Rwanda         31.266933  14.269803    348
Zimbabwe       29.298370  12.418162    316
Vietnam        46.985490   8.509833    446
Zambia         18.692429   5.704748    516
South Africa   24.722701   2.927782    287


In [94]:
# Before concluding destination matters, I need to check whether high-cost countries are just the ones getting shipped to mostly by air, and if so, 
# destination might really just be an indicator of shipment mode. This crosstab shows what percentage of each country's shipments use each mode, 
# so I can see whether mode and destination are intermixed or separate.

In [95]:
mode_by_country = pd.crosstab(df_clean[df_clean['Country'].isin(['Haiti','Rwanda','Zimbabwe','Vietnam','Zambia','South Africa'])]['Country'], 
                                df_clean['Shipment Mode'], normalize='index') * 100
print(mode_by_country.round(1))

Shipment Mode   Air  Air Charter  Ocean  Truck
Country                                       
Haiti          90.4          0.3    9.3    0.0
Rwanda         80.7          0.3    5.5   13.5
South Africa   32.4          0.0   66.9    0.7
Vietnam        99.8          0.0    0.0    0.2
Zambia         38.9          0.0    0.7   60.5
Zimbabwe       31.4          5.6    1.0   62.1


In [96]:
# Same this as shipment mode, but for vendor. I want to check whether certain vendors are tied almost exclusively to certain destinations, 
# since that would also make it hard to tell whether destination or vendor is a real cost real driver.

In [97]:
vendor_country = pd.crosstab(
    df_clean[df_clean['Country'].isin(['Haiti','Rwanda','Zimbabwe','Vietnam','Zambia','South Africa']) & 
             df_clean['Vendor'].isin(top_vendors) & (df_clean['Vendor'] != 'SCMS from RDC')]['Vendor'],
    df_clean['Country']
)
print(vendor_country)

Country                                             Haiti  Rwanda  \
Vendor                                                              
ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV)        32      14   
Aurobindo Pharma Limited                               36      39   
CIPLA LIMITED                                          17      19   
HETERO LABS LIMITED                                    21      30   
MYLAN LABORATORIES LTD (FORMERLY MATRIX LABORAT...     24      24   
Orgenics, Ltd                                          21      15   
Standard Diagnostics, Inc.                              1       3   
Trinity Biotech, Plc                                   15      16   

Country                                             South Africa  Vietnam  \
Vendor                                                                      
ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV)                0       83   
Aurobindo Pharma Limited                                     182       85   
C

In [98]:
# A simple country-level average doesn't prove destination matters on its own, instead 
# it could just mean expensive vendors happen to ship to expensive countries. To actually isolate the destination effect, 
# I picked one vendor (Aurobindo) that ships to many different countries with real volume, 
# and compared its own freight cost across those destinations. If the same vendor's cost still varies a lot by country, 
# there is much stronger evidence that destination itself matters, instead of just vendor pricing.

In [99]:
aurobindo = df_clean[df_clean['Vendor'] == 'Aurobindo Pharma Limited']
aurobindo_by_country = aurobindo.groupby('Country')['freight_pct_of_value'].agg(['median', 'count'])
print(aurobindo_by_country.sort_values('median', ascending=False))

                        median  count
Country                              
Namibia             165.152052      1
Congo, DRC          152.530081      9
Nigeria             140.880749      4
South Sudan         107.188012      2
Angola              104.187748      1
Haiti                70.295897     36
Guyana               40.257298      7
Ghana                37.991551     14
Vietnam              24.995501     85
Côte d'Ivoire        16.899924     16
Zimbabwe             16.619363      9
Uganda               16.486556     19
Cameroon             15.767549     13
Rwanda               12.846639     39
Zambia               11.966721     17
Mozambique           11.340571      2
Tanzania              7.928722     25
Dominican Republic    5.349646      4
Ethiopia              4.951205     12
Swaziland             4.223058      2
South Africa          2.872698    182


In [100]:
# Aurobindo alone is a strong signal, but checking multiple vendors that ship to both Zambia and Zimbabwe 
# specifically gives an even more direct comparison. If several different vendors are all more expensive shipping to Zimbabwe than Zambia, 
# that likely indicates a destination effect rather than a vendor pricing difference.

In [101]:
zam_zim = df_clean[
    df_clean['Country'].isin(['Zambia', 'Zimbabwe']) & 
    df_clean['Vendor'].isin(top_vendors) & (df_clean['Vendor'] != 'SCMS from RDC')
]

vendor_by_country_cost = zam_zim.groupby(['Country', 'Vendor'])['freight_pct_of_value'].agg(['median', 'count'])
print(vendor_by_country_cost)

                                                                median  count
Country  Vendor                                                              
Zambia   ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV)     19.934693      8
         Aurobindo Pharma Limited                            11.966721     17
         CIPLA LIMITED                                        8.422362      2
         HETERO LABS LIMITED                                 15.680261      1
         MYLAN LABORATORIES LTD (FORMERLY MATRIX LABORAT...   6.938100      3
         Orgenics, Ltd                                        9.384251     54
         Standard Diagnostics, Inc.                          17.589969      5
         Trinity Biotech, Plc                                11.082688     25
Zimbabwe ABBVIE LOGISTICS (FORMERLY ABBOTT LOGISTICS BV)     27.041871     26
         Aurobindo Pharma Limited                            16.619363      9
         CIPLA LIMITED                                       32.

In [102]:
# The Zambia vs. Zimbabwe comparison looked meaningfully different, but I wanted to test whether it was actually different or just looked different 
# because some of that gap could just be random variation. I used a Mann-Whitney U test instead of a standard t-test because freight cost 
# is heavily right-skewed throughout this dataset, and a t-test assumes roughly normal data, which this clearly isn't. 
# A p-value below 0.05 means the difference is unlikely to be random chance.

In [103]:
from scipy.stats import mannwhitneyu

zambia_costs = df_clean[df_clean['Country'] == 'Zambia']['freight_pct_of_value']
zimbabwe_costs = df_clean[df_clean['Country'] == 'Zimbabwe']['freight_pct_of_value']

stat, p_value = mannwhitneyu(zambia_costs, zimbabwe_costs, alternative='two-sided')
print(f"Zambia median: {zambia_costs.median():.2f}%, n={len(zambia_costs)}")
print(f"Zimbabwe median: {zimbabwe_costs.median():.2f}%, n={len(zimbabwe_costs)}")
print(f"U statistic: {stat:.1f}")
print(f"p-value: {p_value:.4f}")

Zambia median: 5.70%, n=516
Zimbabwe median: 12.42%, n=316
U statistic: 54116.0
p-value: 0.0000


In [104]:
# The p-value of 0.0000 (below the standard 0.05 threshold) confirms this difference is statistically significant, and 
# with sample sizes this large (516 and 316), it's extremely unlikely this gap between Zambia's 5.7% median and Zimbabwe's 12.4% median 
# happened by chance. Combined with the earlier findings that both countries have a nearly identical shipment mode mix (~60% Truck) 
# and that every vendor shipping to both destinations was consistently more expensive to Zimbabwe, this confirms destination has a real, 
# independent effect on freight cost, that is separate from vendor pricing and shipping mode choice.

In [106]:
#Summary of Findings

#This analysis identified three independent drivers of freight cost as a percentage of shipment value:

#1. Shipment mode — Air freight (Air and Air Charter) consistently costs 2-3x more relative to shipment value than Truck or Ocean, 
# holding true at both the median and 75th percentile.

#2. Vendor — Among high-volume vendors, freight cost ranges from 5.8% (Mylan Laboratories) to 25.0% (ABBVIE Logistics) of shipment value, 
# whcih showed to be a real and consistent difference that wasn't driven by a small sample of outliers.

#3. Destination — Controlling for both vendor and shipment mode, destination showed to have an independent effect on cost. 
# Zimbabwe consistently costs more than Zambia across every vendor shipping to both countries, despite nearly identical shipment mode mixes, 
# and this difference is confirmed statistically significant (Mann-Whitney U test, p < 0.001).

# Practical implication: these three factors don't fully explain each other — 
# a shipment's freight cost depends on the combination of mode, vendor, and destination, not any single one alone. 
# Cost reduction efforts should account for all three separately rather than assuming a single lever 
# (like switching modes, or renegotiating with one vendor) will address cost differences that are actually rooted in 
# destination-specific logistics factors.

# Limitations: this analysis excluded ~40% of shipments where freight cost wasn't directly reported, 
# and the top 1% of remaining records as statistical outliers — findings reflect the ~6,100 shipments with directly attributable freight costs.